In [ ]:
import json
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from deep_translator import GoogleTranslator
from datetime import datetime, timedelta, timezone
import re
import wave
import json
from vosk import Model, KaldiRecognizer
import threading
import wave
import queue

# Declare model path
model_path = r"C:\Users\graem\Downloads\vosk-model-small-en-us-0.15\vosk-model-small-en-us-0.15"
model = Model(model_path)

# Global control flag
global listening
listening = False

global audio_frames
audio_frames = []
q = queue.Queue()

# Supported languages (can be expanded)
languages = {
    "English": "en",
    "Spanish": "es",
    "French": "fr",
    "German": "de",
    "Chinese (Simplified)": "zh-CN",
    "Japanese": "ja",
    "Korean": "ko",
    "Arabic": "ar",
    "Hindi": "hi",
    "Italian": "it"
}

class MedicalApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Electronic Medical Record System")
        self.root.geometry("800x600")

        self.data = {}

        # Style
        style = ttk.Style()
        style.theme_use("clam")

        # Top buttons
        top_frame = ttk.Frame(root, padding=10)
        top_frame.pack(fill="x")

        now = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

        ttk.Button(top_frame, text="Load Record", command=self.load_json).pack(side="left", padx=5)
        ttk.Button(top_frame, text="Save Record", command=self.save_json).pack(side="left", padx=5)
        ttk.Label(top_frame, text=now).pack(side="left", padx=5)

        # Tabs
        self.notebook = ttk.Notebook(root)
        self.notebook.pack(fill="both", expand=True, padx=10, pady=10)

        self.create_patient_tab()
        self.create_medical_tab()
        self.create_nursesnotes_tab()
        self.create_languagetranslator_tab()

    # -------------------------
    # Patient Info Tab
    # -------------------------
    def create_patient_tab(self):
        self.patient_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.patient_tab, text="Patient Details")

        frame = ttk.LabelFrame(self.patient_tab, text="Personal Information", padding=15)
        frame.pack(fill="x", padx=10, pady=10)

        self.fields = {}

        labels = [
            ("Patient ID", "patient_id"),
            ("First Name", "first_name"),
            ("Last Name", "last_name"),
            ("Email", "email"),
            ("Phone", "phone")
        ]

        for i, (label, key) in enumerate(labels):
            ttk.Label(frame, text=label).grid(row=i, column=0, sticky="w", pady=5)
            entry = ttk.Entry(frame, width=40)
            entry.grid(row=i, column=1, pady=5)
            self.fields[key] = entry

        # 👇 Add Gender Combobox (next row)
        gender_row = len(labels)
    
        ttk.Label(frame, text="Gender").grid(row=gender_row, column=0, sticky="w", pady=5)
    
        self.gender_var = tk.StringVar()
        gender_combo = ttk.Combobox(
            frame,
            textvariable=self.gender_var,
            values=["Male", "Female", "Other"],
            state="readonly",
            width=37
        )
        gender_combo.grid(row=gender_row, column=1, pady=5)
        gender_combo.set("Male")  # default value
    
        # Store it like other fields (optional but recommended)
        self.fields["gender"] = gender_combo

    # -------------------------
    # Medical Tab
    # -------------------------
    def create_medical_tab(self):
        self.medical_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.medical_tab, text="Medical Data")

        # Conditions
        cond_frame = ttk.LabelFrame(self.medical_tab, text="Conditions", padding=10)
        cond_frame.pack(fill="both", expand=True, padx=10, pady=5)

        self.conditions_list = tk.Listbox(cond_frame, height=6)
        self.conditions_list.pack(side="left", fill="both", expand=True)

        ttk.Button(cond_frame, text="Add", command=self.add_condition).pack(side="right", padx=5)

        # Medications
        med_frame = ttk.LabelFrame(self.medical_tab, text="Medications", padding=10)
        med_frame.pack(fill="both", expand=True, padx=10, pady=5)

        self.medications_list = tk.Listbox(med_frame, height=6)
        self.medications_list.pack(side="left", fill="both", expand=True)

        ttk.Button(med_frame, text="Add", command=self.add_medication).pack(side="right", padx=5)

    # -------------------------
    # Nurses Notes Tab
    # -------------------------
    def create_nursesnotes_tab(self):
        self.nursesnotes_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.nursesnotes_tab, text="Nursing Notes")
    
        main_frame = ttk.Frame(self.nursesnotes_tab, padding=10)
        main_frame.pack(fill="both", expand=True)

         # Nursing notes text box
        ttk.Label(main_frame, text="Nursing Notes:").pack(anchor="w")
        self.input_text = tk.Text(main_frame, height=10)
        self.input_text.pack(fill="x", pady=5)

        ttk.Button(main_frame, text="Start Recording", command=self.start_listening).pack(side="left", padx=5)
        ttk.Button(main_frame, text="Stop Recording", command=self.stop_listening).pack(side="left", padx=5)
    
    # -------------------------
    # Language Translator Tab
    # -------------------------
    def create_languagetranslator_tab(self):
        self.languagetranslator_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.languagetranslator_tab, text="Language Translator")
    
        main_frame = ttk.Frame(self.languagetranslator_tab, padding=10)
        main_frame.pack(fill="both", expand=True)
    
        # Input Text
        ttk.Label(main_frame, text="Input Text:").pack(anchor="w")
        self.input_text = tk.Text(main_frame, height=6)
        self.input_text.pack(fill="x", pady=5)
    
        # Language selection frame (GRID ONLY here)
        control_frame = ttk.Frame(main_frame)
        control_frame.pack(pady=10)
    
        ttk.Label(control_frame, text="From:").grid(row=0, column=0, padx=5)
        self.from_lang = ttk.Combobox(control_frame, values=list(languages.keys()), state="readonly")
        self.from_lang.grid(row=0, column=1, padx=5)
        self.from_lang.set("English")
    
        ttk.Label(control_frame, text="To:").grid(row=0, column=2, padx=5)
        self.to_lang = ttk.Combobox(control_frame, values=list(languages.keys()), state="readonly")
        self.to_lang.grid(row=0, column=3, padx=5)
        self.to_lang.set("Spanish")
    
        ttk.Button(main_frame, text="Translate", command=self.translate_text).pack(pady=10)
    
        # Output Text
        ttk.Label(main_frame, text="Translated Text:").pack(anchor="w")
        self.output_text = tk.Text(main_frame, height=6)
        self.output_text.pack(fill="x", pady=5)

    # -------------------------
    # Language Translator Tab Button
    # -------------------------
    def translate_text(self):
        try:
            input_value = self.input_text.get("1.0", tk.END).strip()
            if not input_value:
                messagebox.showwarning("Warning", "Please enter text to translate.")
                return

            from_language = languages[self.from_lang.get()]
            to_language = languages[self.to_lang.get()]

            translated = GoogleTranslator(source=from_language, target=to_language).translate(input_value)

            self.output_text.delete("1.0", tk.END)
            self.output_text.insert(tk.END, translated)

        except Exception as e:
            messagebox.showerror("Error", str(e))

    # -------------------------
    # Validation
    # -------------------------
    def validate(self):
        email = self.fields["email"].get()
        phone = self.fields["phone"].get()

        if not re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$', email):
            messagebox.showerror("Error", "Invalid email")
            return False

        if not re.match(r'^\+?\d[\d\- ]+$', phone):
            messagebox.showerror("Error", "Invalid phone")
            return False

        return True

    # -------------------------
    # Load JSON
    # -------------------------
    def load_json(self):
        path = filedialog.askopenfilename(filetypes=[("JSON", "*.json")])
        if not path:
            return

        with open(path) as f:
            self.data = json.load(f)

        # Clear fields
        for field in self.fields.values():
            field.delete(0, tk.END)

        self.conditions_list.delete(0, tk.END)
        self.medications_list.delete(0, tk.END)

        # Populate
        pd = self.data["personal_details"]
        contact = pd["contact"]

        self.fields["patient_id"].insert(0, self.data["patient_id"])
        self.fields["first_name"].insert(0, pd["first_name"])
        self.fields["last_name"].insert(0, pd["last_name"])
        self.fields["email"].insert(0, contact["email"])
        self.fields["phone"].insert(0, contact["phone"])

        for c in self.data["medical_history"]["conditions"]:
            self.conditions_list.insert(tk.END, f"{c['name']} ({c['status']})")

        for m in self.data["medications"]:
            self.medications_list.insert(tk.END, f"{m['name']} - {m['dosage']}")

    # -------------------------
    # Add Condition
    # -------------------------
    def add_condition(self):
        win = tk.Toplevel(self.root)
        win.title("Add Condition")

        ttk.Label(win, text="Name").pack()
        name = ttk.Entry(win)
        name.pack()

        ttk.Label(win, text="Status").pack()
        status = ttk.Entry(win)
        status.pack()

        def save():
            self.conditions_list.insert(tk.END, f"{name.get()} ({status.get()})")
            win.destroy()

        ttk.Button(win, text="Save", command=save).pack()

    # -------------------------
    # Add Medication
    # -------------------------
    def add_medication(self):
        win = tk.Toplevel(self.root)
        win.title("Add Medication")

        ttk.Label(win, text="Name").pack()
        name = ttk.Entry(win)
        name.pack()

        ttk.Label(win, text="Dosage").pack()
        dosage = ttk.Entry(win)
        dosage.pack()

        def save():
            self.medications_list.insert(tk.END, f"{name.get()} - {dosage.get()}")
            win.destroy()

        ttk.Button(win, text="Save", command=save).pack()

    # -------------------------
    # Save JSON
    # -------------------------
    def save_json(self):
        if not self.validate():
            return

        self.data["patient_id"] = self.fields["patient_id"].get()

        pd = self.data["personal_details"]
        contact = pd["contact"]

        pd["first_name"] = self.fields["first_name"].get()
        pd["last_name"] = self.fields["last_name"].get()
        pd["gender"] = self.fields["gender"].get()
        contact["email"] = self.fields["email"].get()
        contact["phone"] = self.fields["phone"].get()

        # Conditions
        conditions = []
        for item in self.conditions_list.get(0, tk.END):
            name, status = item.split(" (")
            conditions.append({
                "name": name,
                "status": status.replace(")", ""),
                "diagnosed_date": ""
            })

        self.data["medical_history"]["conditions"] = conditions

        # Medications
        meds = []
        for item in self.medications_list.get(0, tk.END):
            name, dosage = item.split(" - ")
            meds.append({
                "name": name,
                "dosage": dosage,
                "frequency": "",
                "start_date": ""
            })

        self.data["medications"] = meds

        path = filedialog.asksaveasfilename(defaultextension=".json")
        if not path:
            return

        with open(path, "w") as f:
            json.dump(self.data, f, indent=4)

        messagebox.showinfo("Saved", "Record saved successfully")

    def record():
        # ----------------------------
        # Settings
        # ----------------------------
        duration = 10  # seconds to record
        fs = 16000    # sampling frequency (Hz)
        filename = "recorded_audio.wav"  # output file
    
        # ----------------------------
        # Record audio
        # ----------------------------
        print("Recording...")
        recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
        sd.wait()  # Wait until recording is finished
        print("Recording finished!")
    
        # ----------------------------
        # Save as WAV file
        # ----------------------------
        # Convert to int16 for WAV format
        write(filename, fs, np.int16(recording * 32767))
        print(f"Audio saved as {filename}")
        speech_to_text()

    def speech_to_text():
        # Load the Vosk model
        model_path = r"C:\Users\graem\Downloads\vosk-model-small-en-us-0.15\vosk-model-small-en-us-0.15"
        model = Model(model_path)
    
        # Open recorded audio
        wf = wave.open("recorded.wav", "rb")
    
        # Initialize recognizer
        rec = KaldiRecognizer(model, wf.getframerate())
    
        transcription = ""
    
        # Transcribe
        while True:
            data = wf.readframes(4000)
    
            if len(data) == 0:
                break
    
            if rec.AcceptWaveform(data):
                result = json.loads(rec.Result())
                transcription += result.get("text", "") + " "
    
        # Final result
        result = json.loads(rec.FinalResult())
        transcription += result.get("text", "")
        
        text_notes.insert(tk.END, transcription)
    
        return transcription

    #---------------------------------------------------------------------------------
    # Function for speech
    def audio_thread():
        global listening, audio_frames
    
        rec = KaldiRecognizer(model, 16000)
        audio_frames = []
    
        def callback(indata, frames, time, status):
            if listening:
                data = bytes(indata)
                audio_frames.append(data)
    
                if rec.AcceptWaveform(data):
                    result = json.loads(rec.Result())
                    q.put(result.get("text", ""))
    
        with sd.RawInputStream(samplerate=16000,
                               blocksize=8000,
                               dtype='int16',
                               channels=1,
                               callback=callback):
            while listening:
                sd.sleep(100)   # IMPORTANT: prevents CPU lock

    #---------------------------------------------------------------------------------
    def update_textbox():
        """Safely update UI from queue"""
        try:
            while True:
                text = q.get_nowait()
                text_notes.insert(tk.END, text + "\n")
        except queue.Empty:
            pass
        root.after(100, update_textbox)
    
    #---------------------------------------------------------------------------------
    def save_wav(filename="recorded.wav"):
        wf = wave.open(filename, 'wb')
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(16000)
        wf.writeframes(b''.join(audio_frames))
        wf.close()
    
    #---------------------------------------------------------------------------------
    # Start Recording
    def start_listening():
        global listening
        listening = True
        print("Listening...")
        threading.Thread(target=audio_thread, daemon=True).start()
    
    #---------------------------------------------------------------------------------
    # Stop Recording
    def stop_listening():
        global listening
        listening = False
    
        final = json.loads(KaldiRecognizer(model, 16000).FinalResult())
        q.put("FINAL: " + final.get("text", ""))
        
        save_wav()
        print("Saved to recorded.wav")
    
        speech_to_text()



if __name__ == "__main__":
    root = tk.Tk()
    app = MedicalApp(root)
    root.mainloop()